# Secure Grid Backbone: one working link

*Secure the Grid, step 3 · UTC Problem 6*

The capstone is mostly a design (do the design in your write-up). This notebook is just the **one working link** you need for Part 3.

Pick ONE:
- **Option A:** reuse your BB84 notebook from Step 1 (`1_bb84_eavesdropper.ipynb`) as the link, or
- **Option B:** build the quantum teleportation circuit below (sending one key qubit between nodes).

On a real link, the entangled pair is delivered over fiber, and the two corrections travel as two ordinary classical bits. Here all three qubits are on one chip, so this shows the logic of the link, not its distance.

> **Before you run anything:** check the kernel name at the top right of this notebook. It should be the qBraid kernel (its name includes **qBraid**). If it says **No Kernel**, click it and pick that kernel. Until you do, running a cell does nothing.

In [ ]:
# Use the qBraid SDK environment's kernel. This stops with a clear message if another one is picked.
import importlib.util
missing = [m for m in ("qbraid", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError(f"This kernel has no {', '.join(missing)}. Pick the qBraid SDK environment's "
                      "kernel (Kernel > Change Kernel), then run this cell again.")

# qBraid Lab: install Qiskit Aer if this environment doesn't have it yet (one time, ~20 s)
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()

print("Setup OK: the qBraid kernel and Qiskit Aer are ready. You're good to go.")

In [ ]:
def teleport_circuit():
    """Teleport the state of qubit 0 to qubit 2 using an entangled pair (1,2)."""
    qc = QuantumCircuit(3, 3)
    # Prepare a state to send on qubit 0. It's a "generic" state on purpose:
    # if either correction below is missing, the check in the next cell catches it.
    qc.ry(1.0, 0)
    qc.rz(0.7, 0)

    # TODO: build the teleportation protocol.
    # Hint: entangle qubits 1 and 2 with h(1), cx(1,2);
    #       then cx(0,1), h(0), measure 0 and 1;
    #       then apply cx(1,2) and cz(0,2) corrections. Use plain cx and cz, not if_test:
    #       IonQ can't act on a mid-circuit result, and conditional gates are dropped on the way there.
    #       To check it, undo the state you sent on qubit 2: rz(-0.7, 2), then ry(-1.0, 2),
    #       then measure qubit 2.
    raise NotImplementedError("Build the teleportation link")
    return qc

qc = teleport_circuit()
qc.draw("mpl")


In [ ]:
# Run and show it works
counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
print(counts)

# Qiskit prints bits as (qubit 2)(qubit 1)(qubit 0). Qubits 0 and 1 are random.
# Undoing the state you sent turns qubit 2 back into |0>, so if the link works, it reads 0 on every shot.
measured_q2 = any(i.operation.name == "measure" and qc.find_bit(i.qubits[0]).index == 2 for i in qc.data)
if not measured_q2:
    print("Qubit 2 is not measured yet: undo the state on qubit 2, then measure it (see the hint).")
else:
    zeros = sum(c for bits, c in counts.items() if bits[0] == "0")
    print(f"qubit 2 read 0 on {zeros} of {sum(counts.values())} shots (all of them means the link works)")


In [ ]:
# Run the same link through qBraid on IonQ's free simulator
from qbraid.runtime import QbraidProvider
import warnings
warnings.filterwarnings("ignore", message="Circuit contains measurement gates")  # IonQ measures every qubit anyway

device = QbraidProvider().get_device("ionq:ionq:sim:simulator")
job = device.run(qc, shots=200, runtime_options={"noise_model": "forte-1"})
print("Job ID:", job.id)          # copy this into your write-up
ionq_counts = job.result().data.get_counts()
zeros = sum(c for bits, c in ionq_counts.items() if bits[0] == "0")
print(f"qubit 2 read 0 on {zeros} of {sum(ionq_counts.values())} shots (close to all means the link works)")
# qBraid may warn that measurements are ignored: IonQ measures every qubit at the end, and the
# cx/cz corrections still work because they are applied as quantum gates.


## What to have ready
Written answers go in `WRITEUP.md` in this folder.
- **Part 3:** this notebook (or your reused BB84) running, with a result.
- **Parts 1, 2, 4:** the architecture, assumptions/tradeoffs, and rollout plan — in your write-up. Ground any hardware numbers in the qBraid device list.
